### COMP 3610 Project
#### Feature Selection and Synthetic Data Generation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [2]:
from scipy.stats import ks_2samp
from scipy.stats import wasserstein_distance

In [3]:
df_original = pd.read_csv('Solar Dataset.csv')

In [6]:
df_original.isna().sum()

Day of Year                             0
Year                                    0
Month                                   0
Day                                     0
First Hour of Period                    0
Is Daylight                             0
Distance to Solar Noon                  0
Average Temperature (Day)               0
Average Wind Direction (Day)            0
Average Wind Speed (Day)                0
Sky Cover                               0
Visibility                              0
Relative Humidity                       0
Average Wind Speed (Period)             1
Average Barometric Pressure (Period)    0
Power Generated                         0
dtype: int64

In [28]:
df_original.columns

Index(['Day of Year', 'Year', 'Month', 'Day', 'First Hour of Period',
       'Is Daylight', 'Distance to Solar Noon', 'Average Temperature (Day)',
       'Average Wind Direction (Day)', 'Average Wind Speed (Day)', 'Sky Cover',
       'Visibility', 'Relative Humidity', 'Average Wind Speed (Period)',
       'Average Barometric Pressure (Period)', 'Power Generated'],
      dtype='object')

In [29]:
df_original.head(20)

,Day of Year,Year,Month,Day,First Hour of Period,Is Daylight,Distance to Solar Noon,Average Temperature (Day),Average Wind Direction (Day),Average Wind Speed (Day),Sky Cover,Visibility,Relative Humidity,Average Wind Speed (Period),Average Barometric Pressure (Period),Power Generated
0,245,2023,9,1,1,False,0.859897,69,28,7.5,0,10.0,75,8.0,29.82,0
1,245,2023,9,1,4,False,0.628535,69,28,7.5,0,10.0,77,5.0,29.85,0
2,245,2023,9,1,7,True,0.397172,69,28,7.5,0,10.0,70,0.0,29.89,5418
3,245,2023,9,1,10,True,0.165810,69,28,7.5,0,10.0,33,0.0,29.91,25477
4,245,2023,9,1,13,True,0.065553,69,28,7.5,0,10.0,21,3.0,29.89,30069
5,245,2023,9,1,16,True,0.296915,69,28,7.5,0,10.0,20,23.0,29.85,16280
6,245,2023,9,1,19,True,0.528278,69,28,7.5,0,10.0,36,15.0,29.83,515
7,245,2023,9,1,22,False,0.759640,69,28,7.5,0,10.0,49,6.0,29.86,0
8,246,2023,9,2,1,False,0.862113,72,29,6.8,0,10.0,67,6.0,29.86,0
9,246,2023,9,2,4,False,0.630155,72,29,6.8,0,10.0,49,0.0,29.87,0


In [30]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2920 entries, 0 to 2919
Data columns (total 16 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Day of Year                           2920 non-null   int64  
 1   Year                                  2920 non-null   int64  
 2   Month                                 2920 non-null   int64  
 3   Day                                   2920 non-null   int64  
 4   First Hour of Period                  2920 non-null   int64  
 5   Is Daylight                           2920 non-null   bool   
 6   Distance to Solar Noon                2920 non-null   float64
 7   Average Temperature (Day)             2920 non-null   int64  
 8   Average Wind Direction (Day)          2920 non-null   int64  
 9   Average Wind Speed (Day)              2920 non-null   float64
 10  Sky Cover                             2920 non-null   int64  
 11  Visibility       

Removing null entries and converting bool column to int.

# Step 1 - Preprocessing the data set

## Time features
1) Year - The dataset is take over two years (2023 and 2024). This is not enought values to determine trends over years. Year will be dropped.
2) Month - Very useful as the change in seasons will effect the power generated but it is not as useful as Day of the year.
3) Day - Gives which day of the month it is. Not useful without relation to the month.
4) Day of the year - This is the best way to represent time in the data set as it can capture the change in seasons in a smooth way throughout the year. This will be the feature used to represent date.
5) First hour of period - This will be used as power generation depends on time of day.

## Sine and cosine encoding
Since the 1st day of the year and the last day of the year are adjacent to each other in terms of solar position and simlary the first and last hour of the day. To get better meaning out of these features, sine and cosine encoding will be applied.

## Other features
Some models learn what combination of features over time are useful for the prediction task. Therefore all the other features will be kept in. All the features will be scaled between a specific range so all features will have equal weight.

In [32]:
#Step 1: Sin and Cos encoding
df_original['day_of_year_sin'] = np.sin(2 * np.pi * df_original['Day of Year'] / 365)
df_original['day_of_year_cos'] = np.cos(2 * np.pi * df_original['Day of Year'] / 365)

df_original['hour_sin'] = np.sin(2 * np.pi * df_original['First Hour of Period'] / 24)
df_original['hour_cos'] = np.cos(2 * np.pi * df_original['First Hour of Period'] / 24)

# Step 2: Drop original cyclical features
df_original.drop(['Day of Year', 'First Hour of Period', 'Year', 'Month', 'Day'], axis=1, inplace=True)

numerical_cols =df_original.columns.drop(['day_of_year_sin', 'day_of_year_cos', 'hour_sin', 'hour_cos','Power Generated'])
scaler = MinMaxScaler()
df_original[numerical_cols] = scaler.fit_transform(df_original[numerical_cols])

In [31]:
df_original = df_original.dropna(how = 'any', axis =0)
df_original['Is Daylight'] = df_original['Is Daylight'].astype(int)

In [33]:
df_original.columns

Index(['Is Daylight', 'Distance to Solar Noon', 'Average Temperature (Day)',
       'Average Wind Direction (Day)', 'Average Wind Speed (Day)', 'Sky Cover',
       'Visibility', 'Relative Humidity', 'Average Wind Speed (Period)',
       'Average Barometric Pressure (Period)', 'Power Generated',
       'day_of_year_sin', 'day_of_year_cos', 'hour_sin', 'hour_cos'],
      dtype='object')

### Feature Analysis
We performed correlation analysis using pandas correlation function as well as feature importance analysis using Random Forest.

In [34]:
feature_df = df_original.drop(columns = ['hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos'], axis = 1)
correlation_matrix = feature_df.corr()

x = feature_df.drop(columns=["Power Generated"])  # Features
y = feature_df["Power Generated"]  # Target
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2)
clf = RandomForestClassifier()
clf.fit(x_train, y_train)

feature_importances = clf.feature_importances_
features = x.columns

In [35]:
# Mean Decrease Accuracy
importance = []
acc = accuracy_score(y_test, clf.predict(x_test))
for i in range(x.shape[1]):
    x_test_copy = x_test.copy()
    np.random.shuffle(x_test_copy.iloc[:, i].values)
    shuff_accuracy = accuracy_score(y_test, clf.predict(x_test_copy))
    importance.append(acc - shuff_accuracy)
accuracy_df = pd.DataFrame({'Feature': features, 'Decrease in Accuracy': importance}).sort_values('Decrease in Accuracy', ascending=False)
print(accuracy_df)

                                Feature  Decrease in Accuracy
1                Distance to Solar Noon              0.011986
2             Average Temperature (Day)              0.001712
8           Average Wind Speed (Period)             -0.001712
4              Average Wind Speed (Day)             -0.001712
5                             Sky Cover             -0.001712
6                            Visibility             -0.001712
7                     Relative Humidity             -0.001712
3          Average Wind Direction (Day)             -0.003425
9  Average Barometric Pressure (Period)             -0.003425
0                           Is Daylight             -0.006849


In [21]:
feature_importance_df = pd.DataFrame({"Feature": features, "Importance": feature_importances})
# Sort the features by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by="Importance", ascending=False)

In [22]:
print(correlation_matrix['Power Generated'].sort_values(ascending=False))
print(feature_importance_df)

Power Generated                         1.000000
Is Daylight                             0.532336
Average Wind Speed (Period)             0.278174
Average Wind Direction (Day)            0.146463
Average Wind Speed (Day)                0.142366
Average Temperature (Day)               0.132155
Visibility                              0.075841
Average Barometric Pressure (Period)   -0.036553
Sky Cover                              -0.187248
Relative Humidity                      -0.522445
Distance to Solar Noon                 -0.746825
Name: Power Generated, dtype: float64
                                Feature  Importance
1                Distance to Solar Noon    0.279160
7                     Relative Humidity    0.110192
9  Average Barometric Pressure (Period)    0.104705
4              Average Wind Speed (Day)    0.102385
0                           Is Daylight    0.089178
8           Average Wind Speed (Period)    0.087402
2             Average Temperature (Day)    0.086437
3      

In [ ]:
plt.barh(features, feature_importances, palette = 'magma')
plt.title('Feature Importance - Gini Importance')
plt.show()

In [37]:
df_original = df_original.drop(columns = ['Visibility', 'Average Wind Speed (Day)'], axis = 1)

In [40]:
df_original.to_csv('Original Data Clean.csv', index = False, encoding = 'utf-8')